# 🔐 Senhas em Risco: Uma Análise de Big Data sobre Segurança Digital de Credenciais

---

**Disciplina:** Tópicos de Big Data em Python  
**Instituição:** Unimetrocamp Wyden — Campinas/SP  
**Grupo:**
- Eduardo Gombrade — Análise e Desenvolvimento
- Leandro Schiavo — Documentação
- João Vendito — Dashboard e Visualizações

---

## 📓 Notebook 02 — Limpeza e Tratamento dos Dados

**Objetivo deste notebook:**  
Carregar os dados brutos salvos no Notebook 01, realizar a limpeza,
padronização e enriquecimento de cada dataset, e salvar as versões
tratadas prontas para análise no Notebook 03.

**Etapas executadas:**
1. Remoção de valores nulos e duplicatas
2. Padronização de tipos de dados
3. Criação de features derivadas (engenharia de atributos)
4. Categorização e normalização
5. Validação final dos dados tratados


In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


---
## ⚙️ CÉLULA 1 — Importação das Bibliotecas


In [17]:
# ============================================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# ============================================================

import pandas as pd
import numpy as np
import re
import os
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Bibliotecas importadas!")
print(f"📅 Sessão iniciada em: {datetime.now().strftime('%d/%m/%Y às %H:%M')}")

✅ Bibliotecas importadas!
📅 Sessão iniciada em: 26/05/2026 às 02:09


---
## 📂 CÉLULA 2 — Carregamento dos Dados Brutos

Carregamos os arquivos `.parquet` salvos no Notebook 01.  
Se os `.parquet` não existirem ainda, carregamos direto dos CSVs originais.


In [18]:
# ============================================================
# CARREGAMENTO DOS DADOS — COM FALLBACK PARA CSV
# ============================================================

def carregar_dado(caminho_parquet, caminho_csv, nome, **kwargs):
    """
    Tenta carregar parquet primeiro.
    Se não existir, carrega o CSV como alternativa.
    """
    if os.path.exists(caminho_parquet):
        df = pd.read_parquet(caminho_parquet)
        print(f"✅ {nome}: carregado via Parquet ({len(df):,} linhas)")
    elif os.path.exists(caminho_csv):
        df = pd.read_csv(caminho_csv, encoding='utf-8', on_bad_lines='skip', **kwargs)
        print(f"✅ {nome}: carregado via CSV ({len(df):,} linhas)")
    else:
        print(f"❌ {nome}: arquivo não encontrado em nenhum dos caminhos.")
        return None
    return df

# Carregando cada dataset
df_passwords = carregar_dado(
    'data/processed/passwords_raw.parquet',
    'data/raw/password_strength.csv',
    'Password Strength'
)

df_rockyou = carregar_dado(
    'data/processed/rockyou_raw.parquet',
    'data/raw/rockyou_sample.csv',
    'RockYou Sample'
)

df_breaches = carregar_dado(
    'data/processed/breaches_raw.parquet',
    'data/raw/data_breaches.csv',
    'Data Breaches'
)

df_nordpass = carregar_dado(
    'data/processed/nordpass_raw.parquet',
    'data/external/nordpass_2024.csv',
    'NordPass 2024'
)

print("\n🏁 Todos os datasets carregados!")

✅ Password Strength: carregado via Parquet (669,640 linhas)
✅ RockYou Sample: carregado via Parquet (154 linhas)
✅ Data Breaches: carregado via Parquet (295 linhas)
✅ NordPass 2024: carregado via Parquet (30 linhas)

🏁 Todos os datasets carregados!


---
## 🧹 CÉLULA 3 — Limpeza do Dataset 1: Password Strength

**Problemas comuns neste dataset:**
- Senhas nulas ou vazias
- Registros duplicados
- Coluna `strength` com valores inesperados
- Senhas com caracteres inválidos ou encoding quebrado


In [19]:
# ============================================================
# LIMPEZA — DATASET 1: PASSWORD STRENGTH
# ============================================================

print("📊 ANTES da limpeza:")
print(f"   Linhas       : {len(df_passwords):,}")
print(f"   Nulos        : {df_passwords.isnull().sum().sum():,}")
print(f"   Duplicatas   : {df_passwords.duplicated().sum():,}")

# --- PASSO 1: Renomear colunas para português ---
df_passwords.columns = ['senha', 'forca']

# --- PASSO 2: Remover nulos ---
df_passwords = df_passwords.dropna(subset=['senha'])

# --- PASSO 3: Converter senha para string e remover espaços ---
df_passwords['senha'] = df_passwords['senha'].astype(str).str.strip()

# --- PASSO 4: Remover senhas vazias ou muito curtas (menos de 1 caractere) ---
df_passwords = df_passwords[df_passwords['senha'].str.len() >= 1]

# --- PASSO 5: Remover duplicatas ---
df_passwords = df_passwords.drop_duplicates(subset=['senha']).reset_index(drop=True)

# --- PASSO 6: Garantir que 'forca' só tem valores válidos (0, 1, 2) ---
df_passwords = df_passwords[df_passwords['forca'].isin([0, 1, 2])]

# --- PASSO 7: Mapear força numérica para texto ---
mapa_forca = {0: 'Fraca', 1: 'Média', 2: 'Forte'}
df_passwords['forca_label'] = df_passwords['forca'].map(mapa_forca)

print("\n📊 DEPOIS da limpeza:")
print(f"   Linhas       : {len(df_passwords):,}")
print(f"   Nulos        : {df_passwords.isnull().sum().sum():,}")
print(f"   Duplicatas   : {df_passwords.duplicated().sum():,}")
print("\n🔍 Amostra:")
display(df_passwords.head(8))

📊 ANTES da limpeza:
   Linhas       : 669,640
   Nulos        : 1
   Duplicatas   : 0

📊 DEPOIS da limpeza:
   Linhas       : 669,598
   Nulos        : 0
   Duplicatas   : 0

🔍 Amostra:


,senha,forca,forca_label
0,kzde5577,1,Média
1,kino3434,1,Média
2,visi7k1yr,1,Média
3,megzy123,1,Média
4,lamborghin1,1,Média
5,AVYq1lDE4MgAZfNt,2,Forte
6,u6c8vhow,1,Média
7,v1118714,1,Média


---
## 🔬 CÉLULA 4 — Engenharia de Atributos: Password Strength

**Engenharia de atributos** (Feature Engineering) consiste em criar novas colunas
derivadas dos dados originais para enriquecer a análise.

Vamos extrair das senhas:
- Comprimento
- Presença de letras maiúsculas, minúsculas, números e símbolos
- Tipo de composição
- Tempo estimado para quebra


In [20]:
# ============================================================
# ENGENHARIA DE ATRIBUTOS — DATASET 1: PASSWORD STRENGTH
# ============================================================

# --- Comprimento da senha ---
df_passwords['comprimento'] = df_passwords['senha'].str.len()

# --- Presença de cada tipo de caractere (True/False → 1/0) ---
df_passwords['tem_maiuscula']  = df_passwords['senha'].str.contains(r'[A-Z]', regex=True).astype(int)
df_passwords['tem_minuscula']  = df_passwords['senha'].str.contains(r'[a-z]', regex=True).astype(int)
df_passwords['tem_numero']     = df_passwords['senha'].str.contains(r'[0-9]', regex=True).astype(int)
df_passwords['tem_simbolo']    = df_passwords['senha'].str.contains(r'[^A-Za-z0-9]', regex=True).astype(int)

# --- Contagem de tipos diferentes de caracteres usados ---
df_passwords['tipos_usados'] = (
    df_passwords['tem_maiuscula'] +
    df_passwords['tem_minuscula'] +
    df_passwords['tem_numero'] +
    df_passwords['tem_simbolo']
)

# --- Categoria de comprimento ---
def categorizar_comprimento(n):
    if n <= 6:  return 'Muito Curta (≤6)'
    elif n <= 8:  return 'Curta (7-8)'
    elif n <= 12: return 'Média (9-12)'
    elif n <= 16: return 'Longa (13-16)'
    else:         return 'Muito Longa (>16)'

df_passwords['categoria_comprimento'] = df_passwords['comprimento'].apply(categorizar_comprimento)

# --- Tempo estimado para quebra por força bruta ---
# Baseado em estudos de segurança (Hive Systems Password Table 2024)
def estimar_tempo_quebra(row):
    comp = row['comprimento']
    tipos = row['tipos_usados']

    if comp <= 4:   return 'Instantâneo'
    elif comp <= 6:
        if tipos <= 1: return 'Instantâneo'
        else:          return 'Segundos'
    elif comp <= 8:
        if tipos == 1: return 'Minutos'
        elif tipos == 2: return 'Horas'
        else:          return 'Dias'
    elif comp <= 10:
        if tipos <= 2: return 'Semanas'
        else:          return 'Meses'
    elif comp <= 12:
        if tipos <= 2: return 'Anos'
        else:          return 'Décadas'
    else:
        return 'Séculos'

df_passwords['tempo_quebra'] = df_passwords.apply(estimar_tempo_quebra, axis=1)

# --- Ordem lógica do tempo de quebra (para uso em gráficos) ---
ordem_tempo = [
    'Instantâneo', 'Segundos', 'Minutos', 'Horas',
    'Dias', 'Semanas', 'Meses', 'Anos', 'Décadas', 'Séculos'
]
df_passwords['tempo_quebra'] = pd.Categorical(
    df_passwords['tempo_quebra'], categories=ordem_tempo, ordered=True
)

print("✅ Engenharia de atributos concluída!")
print(f"   Colunas geradas: {list(df_passwords.columns)}")
print(f"\n📊 Distribuição por categoria de comprimento:")
print(df_passwords['categoria_comprimento'].value_counts().to_string())
print(f"\n📊 Distribuição por tempo estimado de quebra:")
print(df_passwords['tempo_quebra'].value_counts().sort_index().to_string())
print("\n🔍 Amostra com novos atributos:")
display(df_passwords.head(6))

✅ Engenharia de atributos concluída!
   Colunas geradas: ['senha', 'forca', 'forca_label', 'comprimento', 'tem_maiuscula', 'tem_minuscula', 'tem_numero', 'tem_simbolo', 'tipos_usados', 'categoria_comprimento', 'tempo_quebra']

📊 Distribuição por categoria de comprimento:
categoria_comprimento
Média (9-12)         364442
Curta (7-8)          161983
Longa (13-16)         96331
Muito Curta (≤6)      40037
Muito Longa (>16)      6805

📊 Distribuição por tempo estimado de quebra:
tempo_quebra
Instantâneo       325
Segundos        39712
Minutos           126
Horas          161767
Dias               90
Semanas        281699
Meses             280
Anos            82288
Décadas           175
Séculos        103136

🔍 Amostra com novos atributos:


,senha,forca,forca_label,comprimento,tem_maiuscula,tem_minuscula,tem_numero,tem_simbolo,tipos_usados,categoria_comprimento,tempo_quebra
0,kzde5577,1,Média,8,0,1,1,0,2,Curta (7-8),Horas
1,kino3434,1,Média,8,0,1,1,0,2,Curta (7-8),Horas
2,visi7k1yr,1,Média,9,0,1,1,0,2,Média (9-12),Semanas
3,megzy123,1,Média,8,0,1,1,0,2,Curta (7-8),Horas
4,lamborghin1,1,Média,11,0,1,1,0,2,Média (9-12),Anos
5,AVYq1lDE4MgAZfNt,2,Forte,16,1,1,1,0,3,Longa (13-16),Séculos


---
## 🧹 CÉLULA 5 — Limpeza do Dataset 2: RockYou Sample


In [21]:
# ============================================================
# LIMPEZA — DATASET 2: ROCKYOU SAMPLE
# ============================================================

print("📊 ANTES da limpeza:")
print(f"   Linhas       : {len(df_rockyou):,}")
print(f"   Nulos        : {df_rockyou.isnull().sum().sum():,}")

# --- PASSO 1: Garantir nomes de colunas padronizados ---
df_rockyou.columns = [col.strip().lower() for col in df_rockyou.columns]

# --- PASSO 2: Remover nulos ---
df_rockyou = df_rockyou.dropna()

# --- PASSO 3: Converter tipos ---
df_rockyou['senha']      = df_rockyou['senha'].astype(str).str.strip()
df_rockyou['frequencia'] = pd.to_numeric(df_rockyou['frequencia'], errors='coerce').fillna(0).astype(int)

# --- PASSO 4: Remover senhas inválidas ---
df_rockyou = df_rockyou[df_rockyou['senha'].str.len() >= 1]
df_rockyou = df_rockyou[df_rockyou['frequencia'] > 0]

# --- PASSO 5: Remover duplicatas, mantendo a maior frequência ---
df_rockyou = df_rockyou.sort_values('frequencia', ascending=False)
df_rockyou = df_rockyou.drop_duplicates(subset=['senha']).reset_index(drop=True)

# --- PASSO 6: Adicionar ranking ---
df_rockyou['ranking'] = df_rockyou.index + 1

# --- PASSO 7: Calcular percentual de uso ---
total_uso = df_rockyou['frequencia'].sum()
df_rockyou['pct_uso'] = (df_rockyou['frequencia'] / total_uso * 100).round(4)

# --- PASSO 8: Comprimento da senha ---
df_rockyou['comprimento'] = df_rockyou['senha'].str.len()

print("\n📊 DEPOIS da limpeza:")
print(f"   Linhas     : {len(df_rockyou):,}")
print(f"   Nulos      : {df_rockyou.isnull().sum().sum():,}")
print("\n🏆 Top 10 senhas mais usadas:")
display(df_rockyou.head(10))

📊 ANTES da limpeza:
   Linhas       : 154
   Nulos        : 0

📊 DEPOIS da limpeza:
   Linhas     : 154
   Nulos      : 0

🏆 Top 10 senhas mais usadas:


,frequencia,senha,ranking,pct_uso,comprimento
0,4420388,123456,1,6.48,6
1,3531135,12345,2,5.17,5
2,2467918,123456789,3,3.62,9
3,1644869,password,4,2.41,8
4,1300009,iloveyou,5,1.90,8
5,1218865,princess,6,1.79,8
6,1078266,1234567,7,1.58,7
7,1057726,rockyou,8,1.55,7
8,1050000,12345678,9,1.54,8
9,1038867,abc123,10,1.52,6


---
## 🧹 CÉLULA 6 — Limpeza do Dataset 3: Data Breaches


In [22]:
# ============================================================
# LIMPEZA — DATASET 3: DATA BREACHES
# ============================================================

print("📊 ANTES da limpeza:")
print(f"   Linhas   : {len(df_breaches):,}")
print(f"   Colunas  : {list(df_breaches.columns)}")
print(f"   Nulos    : {df_breaches.isnull().sum().sum():,}")

# --- PASSO 0: Renomear colunas (inglês → português) ---
# Detecta automaticamente qual padrão de coluna está presente
colunas_atuais = list(df_breaches.columns)

# Mapeamento para o caso do dataset do Kaggle (colunas em inglês)
mapa_colunas_kaggle = {
    'Entity'            : 'empresa',
    'Year'              : 'ano',
    'Records'           : 'registros_afetados',
    'Organization type' : 'setor',
    'Method'            : 'metodo_ataque'
}

# Aplica rename apenas nas colunas que existirem
df_breaches = df_breaches.rename(columns={
    k: v for k, v in mapa_colunas_kaggle.items()
    if k in colunas_atuais
})

# Padroniza qualquer coluna restante (minúscula e sem espaço)
df_breaches.columns = [
    col.strip().lower().replace(' ', '_')
    for col in df_breaches.columns
]

print(f"\n✅ Colunas após renomear: {list(df_breaches.columns)}")

# --- PASSO 1: Remover nulos em colunas críticas ---
df_breaches = df_breaches.dropna(subset=['empresa', 'ano', 'registros_afetados'])

# --- PASSO 2: Converter tipos ---
df_breaches['ano']                = pd.to_numeric(df_breaches['ano'], errors='coerce').astype('Int64')
df_breaches['registros_afetados'] = pd.to_numeric(df_breaches['registros_afetados'], errors='coerce').fillna(0).astype(int)
df_breaches['empresa']            = df_breaches['empresa'].astype(str).str.strip()

# --- PASSO 3: Remover anos inválidos ---
df_breaches = df_breaches[(df_breaches['ano'] >= 2004) & (df_breaches['ano'] <= 2024)]

# --- PASSO 4: Categorizar por escala do vazamento ---
def categorizar_escala(n):
    if n == 0:              return 'Não divulgado'
    elif n < 1_000_000:     return 'Pequeno (<1M)'
    elif n < 100_000_000:   return 'Médio (1M–100M)'
    elif n < 1_000_000_000: return 'Grande (100M–1B)'
    else:                   return 'Catastrófico (>1B)'

df_breaches['escala'] = df_breaches['registros_afetados'].apply(categorizar_escala)

# --- PASSO 5: Criar coluna de década ---
df_breaches['decada'] = (df_breaches['ano'] // 10 * 10).astype(str) + 's'

# --- PASSO 6: Ordenar por ano ---
df_breaches = df_breaches.sort_values('ano').reset_index(drop=True)

print("\n📊 DEPOIS da limpeza:")
print(f"   Linhas   : {len(df_breaches):,}")
print(f"   Nulos    : {df_breaches.isnull().sum().sum():,}")
print(f"\n📊 Distribuição por escala:")
print(df_breaches['escala'].value_counts().to_string())
print("\n🔍 Amostra:")
display(df_breaches.head(8))

📊 ANTES da limpeza:
   Linhas   : 295
   Colunas  : ['Entity', 'Year', 'Records', 'Organization type', 'Method']
   Nulos    : 0

✅ Colunas após renomear: ['empresa', 'ano', 'registros_afetados', 'setor', 'metodo_ataque']

📊 DEPOIS da limpeza:
   Linhas   : 293
   Nulos    : 0

📊 Distribuição por escala:
escala
Médio (1M–100M)       148
Pequeno (<1M)         113
Grande (100M–1B)       31
Catastrófico (>1B)      1

🔍 Amostra:


,empresa,ano,registros_afetados,setor,metodo_ataque,escala,decada
0,Japanet Takata,2004,510000,shopping,inside job,Pequeno (<1M),2000s
1,Automatic Data Processing,2005,125000,financial,poor security,Pequeno (<1M),2000s
2,"CardSystems Solutions Inc. (MasterCard, Visa, ...",2005,40000000,financial,hacked,Médio (1M–100M),2000s
3,Bank of America,2005,1200000,financial,lost / stolen media,Médio (1M–100M),2000s
4,DSW Inc.,2005,1400000,retail,hacked,Médio (1M–100M),2000s
5,Citigroup,2005,3900000,financial,lost / stolen media,Médio (1M–100M),2000s
6,TD Ameritrade,2005,200000,financial,lost / stolen media,Pequeno (<1M),2000s
7,Countrywide Financial Corp,2006,2600000,financial,inside job,Médio (1M–100M),2000s


---
## 🧹 CÉLULA 7 — Limpeza do Dataset 4: NordPass


In [23]:
# ============================================================
# LIMPEZA — DATASET 4: NORDPASS 2024
# ============================================================

# Este dataset já foi criado com estrutura limpa no Notebook 01.
# Fazemos apenas validações e pequenos enriquecimentos.

# --- PASSO 1: Garantir tipos corretos ---
df_nordpass['ranking']           = df_nordpass['ranking'].astype(int)
df_nordpass['senha']             = df_nordpass['senha'].astype(str).str.strip()
df_nordpass['usuarios_afetados'] = pd.to_numeric(df_nordpass['usuarios_afetados'], errors='coerce').fillna(0).astype(int)

# --- PASSO 2: Comprimento das senhas ---
df_nordpass['comprimento'] = df_nordpass['senha'].str.len()

# --- PASSO 3: Verificar se a senha é só numérica ---
df_nordpass['apenas_numeros'] = df_nordpass['senha'].str.isnumeric()

# --- PASSO 4: Verificar se é só letras ---
df_nordpass['apenas_letras'] = df_nordpass['senha'].str.isalpha()

print("✅ Dataset NordPass validado e enriquecido!")
print(f"   Linhas     : {len(df_nordpass)}")
print(f"   Nulos      : {df_nordpass.isnull().sum().sum()}")
print(f"\n📊 Senhas apenas numéricas : {df_nordpass['apenas_numeros'].sum()} de {len(df_nordpass)}")
print(f"📊 Senhas apenas letras    : {df_nordpass['apenas_letras'].sum()} de {len(df_nordpass)}")
print(f"📊 Comprimento médio       : {df_nordpass['comprimento'].mean():.1f} caracteres")
display(df_nordpass)

✅ Dataset NordPass validado e enriquecido!
   Linhas     : 30
   Nulos      : 0

📊 Senhas apenas numéricas : 11 de 30
📊 Senhas apenas letras    : 16 de 30
📊 Comprimento médio       : 6.8 caracteres


,ranking,senha,usuarios_afetados,tempo_para_quebrar,pais_mais_comum,comprimento,apenas_numeros,apenas_letras
0,1,123456,4523011,Menos de 1 segundo,Mundial,6,True,False
1,2,password,3783120,Menos de 1 segundo,Mundial,8,False,True
2,3,123456789,2946877,Menos de 1 segundo,Mundial,9,True,False
3,4,12345678,1897654,Menos de 1 segundo,Mundial,8,True,False
4,5,12345,1678432,Menos de 1 segundo,Mundial,5,True,False
5,6,1234567,1234567,Menos de 1 segundo,Mundial,7,True,False
6,7,iloveyou,1198432,Menos de 1 segundo,Mundial,8,False,True
7,8,111111,1056789,Menos de 1 segundo,Mundial,6,True,False
8,9,123123,987654,Menos de 1 segundo,Mundial,6,True,False
9,10,abc123,876543,Menos de 1 segundo,Brasil,6,False,False


---
## 📊 CÉLULA 8 — Relatório Comparativo de Qualidade

Comparamos o estado dos dados **antes** e **depois** do tratamento,
comprovando a efetividade da limpeza realizada.


In [24]:
# ============================================================
# RELATÓRIO COMPARATIVO DE QUALIDADE DOS DADOS
# ============================================================

print("=" * 65)
print("  📋 RELATÓRIO FINAL DE QUALIDADE DOS DADOS TRATADOS")
print("=" * 65)

datasets_finais = {
    'Password Strength': df_passwords,
    'RockYou Sample'   : df_rockyou,
    'Data Breaches'    : df_breaches,
    'NordPass 2024'    : df_nordpass
}

resumo = []
for nome, df in datasets_finais.items():
    resumo.append({
        'Dataset'        : nome,
        'Linhas'         : f"{len(df):,}",
        'Colunas'        : df.shape[1],
        'Nulos'          : df.isnull().sum().sum(),
        'Duplicatas'     : df.duplicated().sum(),
        'Memória (MB)'   : f"{df.memory_usage(deep=True).sum() / 1024**2:.2f}"
    })

df_resumo = pd.DataFrame(resumo)
display(df_resumo)

print("\n✅ Todos os datasets estão limpos e prontos para análise!")
print("   Próximo passo → Notebook 03: Análise Exploratória")

  📋 RELATÓRIO FINAL DE QUALIDADE DOS DADOS TRATADOS


,Dataset,Linhas,Colunas,Nulos,Duplicatas,Memória (MB)
0,Password Strength,"669,598",11,0,0,170.95
1,RockYou Sample,154,5,0,0,0.01
2,Data Breaches,293,7,0,0,0.10
3,NordPass 2024,30,8,0,0,0.01



✅ Todos os datasets estão limpos e prontos para análise!
   Próximo passo → Notebook 03: Análise Exploratória


---
## 💾 CÉLULA 9 — Salvamento dos Dados Tratados

Salvamos as versões limpas e enriquecidas em `data/processed/`
para que os próximos notebooks não precisem repetir o processamento.


In [25]:
# ============================================================
# SALVAMENTO DOS DADOS TRATADOS
# ============================================================

arquivos_salvar = {
    'data/processed/passwords_clean.parquet' : df_passwords,
    'data/processed/rockyou_clean.parquet'   : df_rockyou,
    'data/processed/breaches_clean.parquet'  : df_breaches,
    'data/processed/nordpass_clean.parquet'  : df_nordpass
}

# Também salvamos CSV para uso externo (documentação, Excel, etc.)
arquivos_csv = {
    'data/processed/passwords_clean.csv' : df_passwords,
    'data/processed/rockyou_clean.csv'   : df_rockyou,
    'data/processed/breaches_clean.csv'  : df_breaches,
    'data/processed/nordpass_clean.csv'  : df_nordpass
}

print("💾 Salvando arquivos Parquet...")
for caminho, df in arquivos_salvar.items():
    df.to_parquet(caminho, index=False)
    print(f"   ✅ {caminho}")

print("\n💾 Salvando arquivos CSV...")
for caminho, df in arquivos_csv.items():
    df.to_csv(caminho, index=False, encoding='utf-8')
    print(f"   ✅ {caminho}")

print("\n🏁 Notebook 02 concluído!")
print("   → Próximo passo: Notebook 03 — Análise Exploratória dos Dados")

💾 Salvando arquivos Parquet...
   ✅ data/processed/passwords_clean.parquet
   ✅ data/processed/rockyou_clean.parquet
   ✅ data/processed/breaches_clean.parquet
   ✅ data/processed/nordpass_clean.parquet

💾 Salvando arquivos CSV...
   ✅ data/processed/passwords_clean.csv
   ✅ data/processed/rockyou_clean.csv
   ✅ data/processed/breaches_clean.csv
   ✅ data/processed/nordpass_clean.csv

🏁 Notebook 02 concluído!
   → Próximo passo: Notebook 03 — Análise Exploratória dos Dados
